In [11]:
train_ds = MRNetDataset(train_df, transforms=train_tfm)
valid_ds = MRNetDataset(valid_df, transforms=valid_tfm)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)

print(f"Train : {len(train_ds)} scans  ->  {len(train_loader)} batches/epoch")
print(f"Valid : {len(valid_ds)} scans  ->  {len(valid_loader)} batches/epoch")
s, c, a, t, _ = next(iter(train_loader))
print(f"Batch shape per plane: {tuple(s.shape)}")


Train : 1130 scans  ->  141 batches/epoch
Valid : 120 scans  ->  15 batches/epoch
Batch shape per plane: (8, 3, 224, 224)


## Section 8 — Model Architecture and Pretrained Weights

### MultiViewViT

The model processes all three MRI planes through a single shared ViT-Small backbone. Each plane independently produces a 384-dimensional CLS token feature vector. The three vectors are concatenated into a 1152-dimensional representation and passed through a 3-layer MLP classification head.

**Parameter breakdown — total: 22,390,403**

| Component | Parameters |
|---|---|
| ViT-Small backbone (12 blocks, embed_dim=384, patch_size=16) | ~21,665,664 |
| Classification head (LayerNorm + 3 Linear layers + Dropout) | ~724,739 |
| **Total** | **22,390,403** |

### Pretrained weights: DINO ViT-S/16

The spec recommends SiT-S self-supervised weights. The SiT-S Google Drive checkpoint was inaccessible programmatically (Drive quota restriction), so DINO ViT-S/16 (Caron et al., 2021) is used instead. DINO is also a self-supervised ViT-Small pretrained on ImageNet without class labels — the same principle as SiT-S. The code falls back to timm supervised ImageNet weights if DINO download also fails.

### Progressive unfreezing schedule

| Stage | Epochs | Trainable parameters |
|---|---|---|
| HEAD only | 1 to 4 | 724,739 |
| TOP-6 backbone blocks + head | 5 to 8 | 11,372,291 |
| Full model | 9 onwards | 22,390,403 |

The learning rate schedule restarts with a short warmup each time new layers are unfrozen.


In [12]:
class MultiViewViT(nn.Module):
    # Multi-view ViT-Small for knee MRI multi-label classification.
    # One shared backbone encodes sagittal, coronal and axial planes independently.
    # CLS outputs [B,384] from each plane are concatenated to [B,1152] and classified.
    # Total parameters: 22,390,403
    #   Backbone (ViT-Small/16, 12 blocks): ~21,665,664
    #   Classification head:                  ~724,739

    def __init__(self, num_classes=CFG.NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.MODEL_NAME, pretrained=False, num_classes=0,
            drop_path_rate=CFG.DROP_PATH_RATE
        )
        D = self.backbone.num_features  # 384 for ViT-Small

        self.head = nn.Sequential(
            nn.LayerNorm(D * 3),
            nn.Linear(D * 3, 512), nn.GELU(), nn.Dropout(CFG.DROPOUT_HEAD1),
            nn.Linear(512, 256),   nn.GELU(), nn.Dropout(CFG.DROPOUT_HEAD2),
            nn.Linear(256, num_classes),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, sag, cor, axi):
        feats = torch.cat([self.backbone(sag),
                           self.backbone(cor),
                           self.backbone(axi)], dim=1)
        return self.head(feats)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_n_blocks(self, n):
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in list(self.backbone.blocks)[-n:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True

    def unfreeze_all(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_param_groups(self, head_lr, bb_early_lr, bb_late_lr, wd):
        blocks       = list(self.backbone.blocks)
        half         = len(blocks) // 2
        early_params = [p for block in blocks[:half] for p in block.parameters()]
        late_params  = [p for block in blocks[half:] for p in block.parameters()]
        extra_early  = (list(self.backbone.patch_embed.parameters()) +
                        [self.backbone.cls_token, self.backbone.pos_embed])
        return [
            {"params": self.head.parameters(),
             "lr": head_lr, "weight_decay": wd},
            {"params": late_params + list(self.backbone.norm.parameters()),
             "lr": bb_late_lr, "weight_decay": wd},
            {"params": early_params + extra_early,
             "lr": bb_early_lr, "weight_decay": wd},
        ]


def initialise_model():
    model = MultiViewViT().to(CFG.DEVICE)
    print("Loading DINO ViT-S/16 self-supervised weights...")
    try:
        dino = torch.hub.load(
            "facebookresearch/dino:main", "dino_vits16",
            pretrained=True, verbose=False
        )
        msg = model.backbone.load_state_dict(dino.state_dict(), strict=False)
        del dino; gc.collect()
        model.pretrained_source = "DINO ViT-S/16 (self-supervised, Caron et al. 2021)"
        print(f"DINO weights loaded  missing={len(msg.missing_keys)}  unexpected={len(msg.unexpected_keys)}")
    except Exception as e:
        print(f"DINO failed ({e}), using timm supervised ImageNet weights...")
        supervised = timm.create_model(
            CFG.MODEL_NAME, pretrained=True, num_classes=0,
            drop_path_rate=CFG.DROP_PATH_RATE
        )
        model.backbone.load_state_dict(supervised.state_dict(), strict=True)
        del supervised
        model.pretrained_source = "timm supervised ImageNet (fallback)"
        print("Supervised ImageNet weights loaded")

    total = sum(p.numel() for p in model.parameters())
    print(f"\nPretrained source : {model.pretrained_source}")
    print(f"Total parameters  : {total:,}")
    print(f"Backbone blocks   : {len(list(model.backbone.blocks))}")
    return model


model = initialise_model()


Loading DINO ViT-S/16 self-supervised weights...
DINO weights loaded  missing=0  unexpected=0

Pretrained source : DINO ViT-S/16 (self-supervised, Caron et al. 2021)
Total parameters  : 22,390,403
Backbone blocks   : 12


## Section 9 — Loss Function and Training Setup

### Focal Loss

Standard Binary Cross-Entropy assigns equal importance to all samples. With 81% of scans labelled abnormal, the model achieves low loss by always predicting abnormal. Focal Loss corrects this:

1. **Focusing factor** `(1 - p_t)^gamma` — reduces gradient from easy correct predictions. With gamma=1.5, a sample predicted at 90% confidence contributes about 7x less gradient than an uncertain sample.

2. **Per-label alpha** — scales loss per label by class rarity. ACL (alpha=0.77) contributes nearly 4x more loss than Abnormal (alpha=0.20) for equivalent prediction errors.

Label smoothing of 0.05 prevents overconfident predictions by softening hard 0/1 targets.

### MixUp

From epoch 5, each batch has a 50% chance of being mixed: two samples are linearly blended with a coefficient from Beta(0.3, 0.3). This creates smoother decision boundaries and reduces overfitting.

### Cosine LR schedule with warmup

LR ramps linearly for 2 epochs then follows a cosine decay. Restarted with a short warmup at epochs 5 and 9 when backbone layers are unfrozen.


In [13]:
class FocalBCELoss(nn.Module):
    # Per-label Focal Loss for multi-label binary classification.
    # Formula: FL = -alpha * (1 - p_t)^gamma * log(p_t)
    # gamma=0 reduces to standard BCE. gamma=1.5 is moderate focus on hard cases.

    def __init__(self, alpha=CFG.FOCAL_ALPHA, gamma=CFG.FOCAL_GAMMA,
                 smoothing=CFG.LABEL_SMOOTH):
        super().__init__()
        self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        self.gamma     = gamma
        self.smoothing = smoothing

    def forward(self, logits, targets):
        t       = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        bce     = F.binary_cross_entropy_with_logits(logits, t, reduction="none")
        p_t     = torch.exp(-bce)
        focal_w = (1 - p_t) ** self.gamma
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * focal_w * bce).mean()


def mixup_batch(sag, cor, axi, targets, alpha):
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(sag.size(0))
    return (lam*sag + (1-lam)*sag[perm],
            lam*cor + (1-lam)*cor[perm],
            lam*axi + (1-lam)*axi[perm],
            lam*targets + (1-lam)*targets[perm])


def make_scheduler(optimizer, warmup_ep, total_ep):
    def lr_lambda(ep):
        if ep < warmup_ep:
            return (ep + 1) / warmup_ep
        progress = (ep - warmup_ep) / max(1, total_ep - warmup_ep)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


criterion = FocalBCELoss().to(CFG.DEVICE)
model.freeze_backbone()

optimizer = torch.optim.AdamW(
    model.get_param_groups(CFG.HEAD_LR, CFG.BACKBONE_LR_EARLY,
                           CFG.BACKBONE_LR_LATE, CFG.WEIGHT_DECAY)
)
scheduler = make_scheduler(optimizer, CFG.WARMUP_EPOCHS, CFG.EPOCHS)
writer    = SummaryWriter(CFG.TENSORBOARD_DIR)

print(f"Loss     : FocalBCELoss  gamma={CFG.FOCAL_GAMMA}  alpha={CFG.FOCAL_ALPHA}")
print(f"Backbone : FROZEN  (trainable: {model.trainable_params():,})")
print(f"LR       : head={CFG.HEAD_LR}  bb_late={CFG.BACKBONE_LR_LATE}  bb_early={CFG.BACKBONE_LR_EARLY}")


Loss     : FocalBCELoss  gamma=1.5  alpha=[0.77, 0.63, 0.2]
Backbone : FROZEN  (trainable: 724,739)
LR       : head=0.0003  bb_late=2e-05  bb_early=5e-06


In [14]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch, writer):
    model.train()
    running_loss = 0.
    all_preds, all_targets = [], []
    use_mixup = (epoch >= CFG.MIXUP_START_EP)

    for sag, cor, axi, targets, _ in loader:
        sag, cor, axi = sag.to(device), cor.to(device), axi.to(device)
        targets = targets.to(device)

        if use_mixup and random.random() < CFG.MIXUP_PROB:
            sag, cor, axi, targets = mixup_batch(sag, cor, axi, targets, CFG.MIXUP_ALPHA)

        optimizer.zero_grad(set_to_none=True)
        logits = model(sag, cor, axi)
        loss   = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * sag.size(0)
        with torch.no_grad():
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append((probs >= 0.5).astype(int))
            all_targets.append(targets.detach().cpu().numpy().round().astype(int))

    all_preds   = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    epoch_loss  = running_loss / len(loader.dataset)
    epoch_f1    = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    writer.add_scalar("Loss/train", epoch_loss, epoch)
    writer.add_scalar("F1/train",   epoch_f1,   epoch)
    return epoch_loss, epoch_f1


@torch.no_grad()
def validate(model, loader, criterion, device, epoch, writer):
    model.eval()
    running_loss = 0.
    all_probs, all_preds, all_targets = [], [], []

    for sag, cor, axi, targets, _ in loader:
        sag, cor, axi = sag.to(device), cor.to(device), axi.to(device)
        targets = targets.to(device)
        logits  = model(sag, cor, axi)
        running_loss += criterion(logits, targets).item() * sag.size(0)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_preds.append((probs >= 0.5).astype(int))
        all_targets.append(targets.cpu().numpy())

    all_probs   = np.vstack(all_probs)
    all_preds   = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    epoch_loss  = running_loss / len(loader.dataset)
    epoch_f1    = f1_score(all_targets, all_preds, average="macro", zero_division=0)

    aucs = {}
    for i, name in enumerate(CFG.LABEL_NAMES):
        try:    aucs[name] = roc_auc_score(all_targets[:, i], all_probs[:, i])
        except: aucs[name] = float("nan")

    writer.add_scalar("Loss/val", epoch_loss, epoch)
    writer.add_scalar("F1/val",   epoch_f1,   epoch)
    for name, auc in aucs.items():
        writer.add_scalar(f"AUC/{name}", auc, epoch)
    writer.add_scalar("LR/head",    optimizer.param_groups[0]["lr"], epoch)
    writer.add_scalar("LR/bb_late", optimizer.param_groups[1]["lr"], epoch)
    return epoch_loss, epoch_f1, aucs, all_targets, all_probs, all_preds

print("Training functions ready")


Training functions ready


## Section 10 — Training Loop

Runs training for up to 45 epochs across three progressive unfreezing stages.

**Early stopping** monitors a 3-epoch rolling average of validation F1. Training stops if this does not improve for 12 consecutive epochs after at least 20 epochs.

**Training log columns:**
- `Stage` — HEAD / TOP6 / FULL shows which parameters are being trained
- `TrLoss`, `TrF1` — training loss and macro F1
- `VaLoss`, `VaF1` — validation loss and macro F1
- `ACL`, `MEN`, `ABN` — per-label AUC-ROC on validation
- `Mix` — whether MixUp is active this epoch


In [15]:
history = {k: [] for k in ["train_loss", "train_f1", "val_loss", "val_f1",
                            "auc_acl", "auc_meniscus", "auc_abnormal",
                            "lr_head", "lr_bb_late", "trainable"]}
best_val_f1  = -1.
patience_cnt = 0
recent_f1    = deque(maxlen=3)

print(f"{'Ep':>3} {'Stage':<8} {'TrLoss':>8} {'TrF1':>7} {'VaLoss':>8} "
      f"{'VaF1':>7} {'ACL':>7} {'MEN':>7} {'ABN':>7} {'Mix':>4} {'*':>2}")
print("-" * 90)

for epoch in range(1, CFG.EPOCHS + 1):

    if epoch == CFG.UNFREEZE_PARTIAL_EP + 1:
        model.unfreeze_last_n_blocks(CFG.N_BLOCKS_PARTIAL)
        scheduler = make_scheduler(optimizer, CFG.WARMUP_RESTART, CFG.EPOCHS - epoch)
        print(f"  Ep{epoch}: Unfroze last {CFG.N_BLOCKS_PARTIAL} blocks "
              f"(trainable: {model.trainable_params():,}) -- LR restart")
    elif epoch == CFG.UNFREEZE_FULL_EP + 1:
        model.unfreeze_all()
        scheduler = make_scheduler(optimizer, CFG.WARMUP_RESTART, CFG.EPOCHS - epoch)
        print(f"  Ep{epoch}: Full backbone unfrozen "
              f"(trainable: {model.trainable_params():,}) -- LR restart")

    stage   = ("HEAD"   if epoch <= CFG.UNFREEZE_PARTIAL_EP else
               f"TOP{CFG.N_BLOCKS_PARTIAL}" if epoch <= CFG.UNFREEZE_FULL_EP else "FULL")
    use_mix = epoch >= CFG.MIXUP_START_EP

    train_loss, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, CFG.DEVICE, epoch, writer)
    val_loss, val_f1, aucs, _, _, _ = validate(
        model, valid_loader, criterion, CFG.DEVICE, epoch, writer)
    scheduler.step()

    history["train_loss"].append(train_loss); history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss);     history["val_f1"].append(val_f1)
    for name in CFG.LABEL_NAMES:
        history[f"auc_{name}"].append(aucs[name])
    history["lr_head"].append(optimizer.param_groups[0]["lr"])
    history["lr_bb_late"].append(optimizer.param_groups[1]["lr"])
    history["trainable"].append(model.trainable_params())

    recent_f1.append(val_f1)
    smooth_f1 = np.mean(recent_f1)
    is_best   = smooth_f1 > best_val_f1

    if is_best:
        best_val_f1 = smooth_f1
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "val_f1": val_f1, "aucs": aucs, "history": history},
                   CFG.CHECKPOINT_PATH)
        patience_cnt = 0
    else:
        patience_cnt += 1

    mix_str = "ON" if use_mix else "off"
    print(f"{epoch:>3} {stage:<8} {train_loss:>8.4f} {train_f1:>7.4f} "
          f"{val_loss:>8.4f} {val_f1:>7.4f} {aucs['acl']:>7.4f} "
          f"{aucs['meniscus']:>7.4f} {aucs['abnormal']:>7.4f} "
          f"{mix_str:>4} {'*' if is_best else '':>2}")

    if epoch >= CFG.MIN_EPOCHS and patience_cnt >= CFG.PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

writer.close()
print(f"\nBest smoothed val F1: {best_val_f1:.4f} -- checkpoint saved")


 Ep Stage      TrLoss    TrF1   VaLoss    VaF1     ACL     MEN     ABN  Mix  *
------------------------------------------------------------------------------------------
  1 HEAD       0.1171  0.6652   0.0887  0.7270  0.7023  0.7534  0.8337  off  *
  2 HEAD       0.0931  0.6979   0.0891  0.6979  0.7649  0.6867  0.8097  off   
  3 HEAD       0.0842  0.7293   0.0913  0.7437  0.7854  0.6861  0.8261  off   
  4 HEAD       0.0806  0.7407   0.0879  0.7368  0.8042  0.7050  0.8291  off   
  Ep5: Unfroze last 6 blocks (trainable: 11,372,291) -- LR restart
  5 TOP6       0.0834  0.7284   0.0937  0.7617  0.7952  0.6926  0.8253   ON  *
  6 TOP6       0.0838  0.7433   0.0865  0.7468  0.7781  0.7475  0.8307   ON  *
  7 TOP6       0.0808  0.7616   0.1190  0.6725  0.7396  0.7206  0.8615   ON   
  8 TOP6       0.0764  0.7722   0.0834  0.7660  0.8272  0.7851  0.8754   ON   
  Ep9: Full backbone unfrozen (trainable: 22,390,403) -- LR restart
  9 FULL       0.0692  0.8002   0.0876  0.7794  0.8535  0.7757 